# Sleep disorder correlations (NHANES questionnaire)

Dieses Notebook berechnet **echte bivariate Zusammenhänge** zwischen `sleep_disorder` und den übrigen Variablen im Datensatz.

Es unterscheidet dabei zwischen:

- **kategorialen Features** → `phi` / `cramers_v`
- **numerischen Features** → `point_biserial_r` und optional `spearman_r`

Außerdem enthält es eine kurze **Target-Audit-Sektion**, weil `sleep_disorder` vermutlich relativ **großzügig** definiert wurde.

## Wichtiger methodischer Hinweis

Wenn `sleep_disorder` aus Variablen wie `SLQ040`, `SLQ050` oder RXD-Textfeldern abgeleitet wurde, dann dürfen diese Variablen **nicht** als normale Prädiktoren interpretiert werden — das wäre **Target Leakage**.

Dieses Notebook markiert solche Spalten deshalb separat.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chi2_contingency, pointbiserialr, spearmanr

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
# Pfad anpassen falls nötig
DATA_PATH = Path("merged_questionnaire.csv")

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.columns[:20].tolist())


## 1) Target prüfen

Hier schauen wir uns Prävalenz und mögliche Herkunft der Zielvariable an.


In [ ]:
target = "sleep_disorder"

assert target in df.columns, f"{target!r} nicht im Datensatz gefunden"

print("sleep_disorder value counts:")
print(df[target].value_counts(dropna=False).sort_index())
print()
print("sleep_disorder share:")
print(df[target].value_counts(normalize=True, dropna=False).sort_index().round(4))


In [ ]:
for col, label in [
    ("slq040___how_often_do_you_snort_or_stop_breathing", "SLQ040"),
    ("slq050___ever_told_doctor_had_trouble_sleeping?", "SLQ050"),
]:
    if col in df.columns:
        print(f"{label} value counts:")
        print(df[col].value_counts(dropna=False).sort_index())
        print()

if "rxd_disease_list" in df.columns:
    rxd_sleep = df["rxd_disease_list"].astype(str).str.contains("Insomnia|Sleep disorder", case=False, na=False)
    print(f"People with Insomnia/Sleep disorder in RXD: {int(rxd_sleep.sum()):,}")


## 2) Leakage-Spalten definieren

Diese Variablen wurden wahrscheinlich direkt oder indirekt zur Bildung des Targets verwendet.
Sie werden später **separat** ausgewiesen.


In [ ]:
leakage_cols = {
    "sleep_disorder",
    "slq040___how_often_do_you_snort_or_stop_breathing",
    "slq050___ever_told_doctor_had_trouble_sleeping?",
    "rxd_disease_list",
}

present_leakage_cols = [c for c in leakage_cols if c in df.columns]
present_leakage_cols


## 3) Hilfsfunktionen für Zusammenhänge

### Für kategoriale Variablen
- 2x2 → **phi**
- größere Kreuztabellen → **Cramér's V**

### Für numerische Variablen
- **point-biserial r** gegenüber binärem Target
- optional **Spearman** auf numerisch codierten Werten


In [ ]:
def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.empty or min(tab.shape) < 2:
        return np.nan, np.nan, tab.shape

    chi2, p, _, expected = chi2_contingency(tab)
    n = tab.to_numpy().sum()
    if n == 0:
        return np.nan, np.nan, tab.shape

    r, k = tab.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1)) if n > 1 else 0
    rcorr = r - ((r - 1) ** 2) / (n - 1) if n > 1 else r
    kcorr = k - ((k - 1) ** 2) / (n - 1) if n > 1 else k
    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return np.nan, p, tab.shape
    return np.sqrt(phi2corr / denom), p, tab.shape

def phi_binary(x, y):
    tab = pd.crosstab(x, y)
    if tab.shape != (2, 2):
        return np.nan, np.nan, tab.shape
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.to_numpy().sum()
    phi = np.sqrt(chi2 / n) if n else np.nan
    return phi, p, tab.shape

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    return len(vals) == 2

def maybe_numeric_feature(s, min_non_null=100):
    num = safe_numeric(s)
    return num.notna().sum() >= min_non_null

def classify_feature(s, unique_threshold=12):
    num = safe_numeric(s)
    non_null = num.notna().sum()

    if non_null == 0:
        return "all_missing"

    nunique_num = num.dropna().nunique()
    nunique_raw = s.dropna().nunique()

    # Viele NHANES-Fragen sind numerisch codierte Kategorien
    if nunique_num <= unique_threshold:
        return "categorical"
    if pd.api.types.is_numeric_dtype(s) or non_null >= 0.9 * s.notna().sum():
        return "numeric"
    if nunique_raw <= unique_threshold:
        return "categorical"
    return "categorical"


## 4) Kategoriale Features auswerten


In [ ]:
cat_rows = []

for col in df.columns:
    if col == target:
        continue

    s = df[col]
    feature_type = classify_feature(s)

    if feature_type != "categorical":
        continue

    tmp = pd.DataFrame({"x": s, "y": df[target]}).dropna()
    if len(tmp) < 100:
        continue
    if tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2:
        continue

    if tmp["x"].nunique() == 2 and tmp["y"].nunique() == 2:
        stat, p, shape = phi_binary(tmp["x"], tmp["y"])
        metric = "phi"
    else:
        stat, p, shape = cramers_v(tmp["x"], tmp["y"])
        metric = "cramers_v"

    cat_rows.append({
        "feature": col,
        "metric": metric,
        "association": stat,
        "abs_association": abs(stat) if pd.notna(stat) else np.nan,
        "p_value": p,
        "n": len(tmp),
        "n_unique": tmp["x"].nunique(),
        "table_shape": str(shape),
        "missing_pct": round(df[col].isna().mean() * 100, 2),
        "leakage": col in present_leakage_cols,
    })

cat_results = pd.DataFrame(cat_rows).sort_values(["leakage", "abs_association"], ascending=[True, False])
cat_results.head(30)


## 5) Numerische Features auswerten


In [ ]:
num_rows = []

y = safe_numeric(df[target])

for col in df.columns:
    if col == target:
        continue

    if classify_feature(df[col]) != "numeric":
        continue

    x = safe_numeric(df[col])
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()

    if len(tmp) < 100:
        continue
    if tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2:
        continue

    try:
        r_pb, p_pb = pointbiserialr(tmp["y"], tmp["x"])
    except Exception:
        r_pb, p_pb = np.nan, np.nan

    try:
        r_sp, p_sp = spearmanr(tmp["x"], tmp["y"], nan_policy="omit")
    except Exception:
        r_sp, p_sp = np.nan, np.nan

    num_rows.append({
        "feature": col,
        "point_biserial_r": r_pb,
        "abs_point_biserial_r": abs(r_pb) if pd.notna(r_pb) else np.nan,
        "p_value_point_biserial": p_pb,
        "spearman_r": r_sp,
        "abs_spearman_r": abs(r_sp) if pd.notna(r_sp) else np.nan,
        "p_value_spearman": p_sp,
        "n": len(tmp),
        "n_unique": tmp["x"].nunique(),
        "missing_pct": round(df[col].isna().mean() * 100, 2),
        "leakage": col in present_leakage_cols,
    })

num_results = pd.DataFrame(num_rows).sort_values(["leakage", "abs_point_biserial_r"], ascending=[True, False])
num_results.head(30)


## 6) Top-Features ohne Leakage

Das ist die wichtigere Liste für euer Quiz / Modell.


In [ ]:
top_cat_no_leak = (
    cat_results.loc[~cat_results["leakage"]]
    .sort_values("abs_association", ascending=False)
    .head(30)
)

top_num_no_leak = (
    num_results.loc[~num_results["leakage"]]
    .sort_values("abs_point_biserial_r", ascending=False)
    .head(30)
)

print("Top categorical features (no leakage):")
display(top_cat_no_leak)

print("\nTop numeric features (no leakage):")
display(top_num_no_leak)


## 7) Optional: vereinheitlichtes Ranking

Für eine erste Exploration kann man kategoriale und numerische Features in eine gemeinsame Tabelle schreiben.
Achtung: Die Kennzahlen sind **nicht perfekt direkt vergleichbar**, aber für ein grobes Screening oft praktisch.


In [ ]:
cat_rank = cat_results.assign(score=cat_results["abs_association"], stat_name=cat_results["metric"])
num_rank = num_results.assign(score=num_results["abs_point_biserial_r"], stat_name="point_biserial_r")

common_cols = ["feature", "score", "stat_name", "n", "n_unique", "missing_pct", "leakage"]

combined_rank = pd.concat([
    cat_rank[common_cols],
    num_rank[common_cols]
], ignore_index=True).sort_values(["leakage", "score"], ascending=[True, False])

combined_rank.head(50)


## 8) Alternative, strengere Target-Definitionen prüfen

Eure aktuelle Definition wirkt wahrscheinlich **großzügig**, besonders wenn schon einzelne Hinweise aus `SLQ040` oder RXD genügen.

Darum bauen wir hier zwei optionale Varianten:

- **broad**: bestehendes `sleep_disorder`
- **strict**: eher ärztlich / diagnostisch naheliegende Positiven
- **very_strict**: nur sehr starke Hinweise

> Die genauen Cutoffs müsst ihr fachlich prüfen. Diese Zellen sind als Startpunkt gedacht.


In [ ]:
alt = pd.DataFrame(index=df.index)

# broad = bestehendes Target
alt["sleep_disorder_broad"] = safe_numeric(df[target]) if target in df.columns else np.nan

# Hilfsvariablen
slq040 = safe_numeric(df["slq040___how_often_do_you_snort_or_stop_breathing"]) if "slq040___how_often_do_you_snort_or_stop_breathing" in df.columns else pd.Series(np.nan, index=df.index)
slq050 = safe_numeric(df["slq050___ever_told_doctor_had_trouble_sleeping?"]) if "slq050___ever_told_doctor_had_trouble_sleeping?" in df.columns else pd.Series(np.nan, index=df.index)

if "rxd_disease_list" in df.columns:
    rxd_sleep = df["rxd_disease_list"].astype(str).str.contains("Insomnia|Sleep disorder", case=False, na=False)
else:
    rxd_sleep = pd.Series(False, index=df.index)

# Annahme:
# - NHANES-codierte Ja/Nein-Fragen sind oft 1=Yes, 2=No
# - Häufigkeitsfragen haben oft kleine numerische Codes; kleinere Werte stehen häufig für "häufiger"
#
# Diese Regeln bitte gegen euer Codebook prüfen.
alt["sleep_disorder_strict"] = (
    (slq050 == 1) |
    (rxd_sleep)
).astype("Int64")

alt["sleep_disorder_very_strict"] = (
    (slq050 == 1) |
    (rxd_sleep & slq040.notna())
).astype("Int64")

for c in alt.columns:
    print(c)
    print(alt[c].value_counts(dropna=False).sort_index())
    print()


## 9) CSV-Export der Rankings


In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

cat_results.to_csv(out_dir / "sleep_disorder_categorical_associations.csv", index=False)
num_results.to_csv(out_dir / "sleep_disorder_numeric_associations.csv", index=False)
combined_rank.to_csv(out_dir / "sleep_disorder_combined_ranking.csv", index=False)

print("Saved:")
for p in sorted(out_dir.glob("sleep_disorder_*")):
    print("-", p)


## Interpretation

- Nutzt für euer **Quiz** vor allem die Features mit:
  - hoher Assoziation
  - wenig Missingness
  - guter inhaltlicher Interpretierbarkeit
  - **ohne Leakage**
- Nehmt nicht automatisch die top 10. Manche Variablen sind zwar stark, aber im Quiz unpraktisch oder klinisch redundant.
- Für die spätere Modellierung solltet ihr zusätzlich prüfen:
  - univariate Signalstärke
  - Multikollinearität
  - Fairness / Verzerrung
  - wie gut sich eine Frage in Alltagssprache stellen lässt
